In [1]:
import os
from nwtrace import *
import pandas as pd
import geopandas as gpd

network_path = "data/more/full_sewers.geojson"
node_path = "data/more/full_nodes.geojson"
project_crs = "EPSG:26717"

multiple = True
upstream_only = True
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROM'
downstream_field = 'TO'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

In [2]:

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls
# target_endpoints = ["OF3729806115"]

outputname_extra = "BC_repaired_"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

In [3]:
sewers = gpd.read_file(network_path)
nodes = gpd.read_file(node_path).rename(columns={"FACILITYID": 'node_id'})

In [4]:
repaired_network = repair.repair_segment_connections(
    segments=sewers,
    nodes=nodes,
    segment_id_field=sewer_id_field,
    node_id_field='node_id',
    upstream_field=upstream_field,
    downstream_field=downstream_field,
    distance_threshold=0.1
)

In [22]:
repaired_network = repaired_network.drop_duplicates(subset=sewer_id_field)

In [26]:

sewershed = NWTrace(
    network=repaired_network,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
    crs=project_crs
)

In [27]:
fittings = gpd.read_file("data/more/fitting_connections.geojson")
# catchbasin_leads = gpd.read_file("data/more/catchbasin_leads.gpkg")

In [28]:

# # additional connections
nodes_up = (fittings[["FACILITYID", "TO_FIXED", "geometry"]]
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'})
            .set_index('node_id', drop=False).to_dict(orient="index"))

sewershed.add_upstream_nodes(nodes_up, search_geometry=True)


# new_segs = (catchbasin_leads[["FACILITYID", "UP_ASSET_ID", "DN_ASSET_ID"]]
#             .rename(columns={"FACILITYID": 'segment_id', "UP_ASSET_ID": 'from', "DN_ASSET_ID": 'to'})
#             .set_index('segment_id').to_dict(orient="index"))

# sewershed.add_segments(new_segs)

Added 5501 node-segment connection(s)
Created 540 new node(s)
Created 374 new segment(s).



In [29]:
if multiple == False:
    result = sewershed.trace_sewershed(
        target_endpoints[0], 
        upstream_only=upstream_only, 
        downstream_only=downstream_only
    )
else:
    result = sewershed.trace_sewersheds(
        target_endpoints, 
        upstream_only=upstream_only, 
        downstream_only=downstream_only, 
    )

Tracing Sewer Network from endpoint(s) [OF3783406347(...)]
	Direction(s): upstream
Preparing directional node connection tree...
Searching Network:


100%|██████████| 190/190 [00:00<00:00, 5508.67it/s]


Found 15 connections overall to all 190 endpoints
Finished!


In [30]:
d_node, d_seg = sewershed.get_directional_lookup_tables()

In [31]:
d_seg['CL9478']

{'from': ['CB3838007097'], 'to': ['MH3836707104']}

In [32]:
d_node['MH3873708965']

{'in': ['SL51551', 'SL51542', 'SL53208'], 'out': ['SL53230']}

In [67]:
sewershed_network = pd.DataFrame.from_dict(result).rename(columns={'segment_id': sewer_id_field})
delin_sewershed_network = repaired_network.merge(sewershed_network, on=sewer_id_field, how="right")

delin_sewershed_network.to_file(
    f'{output_dir}/{outputname_extra}catchment_'
    f'{"singledir" if upstream_only or downstream_only else "multidir"}'
    f'{"_ups" if upstream_only and not downstream_only else ""}'
    f'{"_dwns" if downstream_only and not upstream_only else ""}_'
    f'{target_endpoints[0] if not multiple else outfall_file.replace("/", "_").replace(".", "_")}.geojson'
)

KeyError: "None of ['FACILITYID'] are in the columns"